In [ ]:
%pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph langchain-openai langchain-core pypdf unstructured

In [ ]:
# configuração chatgpt
import getpass
import os
from google.colab import userdata
from langchain.chat_models import init_chat_model
from langchain_core.vectorstores import InMemoryVectorStore

In [ ]:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
llm = init_chat_model("gpt-4o-mini", model_provider="openai")

In [ ]:
#selecionando o embedding
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

In [ ]:
vector_store = InMemoryVectorStore(embeddings)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
import os

destino = "/content/drive/MyDrive/chatbot_docs"
os.makedirs(destino, exist_ok=True)

arquivos = [
    "/content/GW_HCA-G2_User-Manual-PT.pdf",
    "/content/goodwe_carregador.pdf",
    "/content/GoodWe_EV_ChargeOps_Base_Conhecimento.pdf",
    "/content/goodwe_instalacao.pdf",
    "/content/goodwe_manualcarregador.pdf",
    "/content/goodwe_manualdousuario.pdf",
    "/content/goodwe_carregador1.pdf"
]

for arquivo in arquivos:
  if os.path.exists(arquivo):
    shutil.move(arquivo, destino)

print("Arquivos movidos com sucesso!")

In [ ]:
#base de cohecimento do chatbot
import bs4
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader

BASE_PATH = "/content/drive/MyDrive/chatbot_docs/"

file_path1 = BASE_PATH + "goodwe_manualdousuario.pdf"
file_path2 = BASE_PATH + "goodwe_carregador.pdf"
file_path3 = BASE_PATH + "goodwe_carregador1.pdf"
file_path4 = BASE_PATH + "goodwe_instalacao.pdf"
file_path5 = BASE_PATH + "goodwe_manualcarregador.pdf"
file_path6 = BASE_PATH + "GW_HCA-G2_User-Manual-PT.pdf"
file_path7 = BASE_PATH + "GoodWe_EV_ChargeOps_Base_Conhecimento.pdf"

loader1 = PyPDFLoader(file_path1)
loader2 = PyPDFLoader(file_path2)
loader3 = WebBaseLoader(["https://br.goodwe.com/"])
loader4 = PyPDFLoader(file_path3)
loader5 = PyPDFLoader(file_path4)
loader6 = PyPDFLoader(file_path5)
loader7 = PyPDFLoader(file_path6)
loader8 = PyPDFLoader(file_path7)

docs1 = loader1.load()
docs2 = loader2.load()
docs3 = loader3.load()
docs4 = loader4.load()
docs5= loader5.load()
docs6 = loader6.load()
docs7 = loader7.load()
docs8 = loader8.load()

docs = docs1 + docs2 + docs3 + docs4 + docs5 + docs6 + docs7 + docs8


In [ ]:
docs

In [ ]:
#splitting dos documentos
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split pdf into {len(all_splits)} sub-documents.")

In [ ]:
all_splits[0]


In [ ]:
#guardando os dados em um banco de dados
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

In [ ]:

from langsmith import Client
client = Client(api_key=userdata.get('LANGSMITH_API_KEY'))
prompt = client.pull_prompt("rlm/rag-prompt", include_model=True)

In [ ]:
prompt

In [ ]:
# Atualizando o prompt

prompt.messages[0].prompt.template = '''You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say 'NAO SEI'. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:'''

In [ ]:
from langchain_core.documents import Document
from typing_extensions import List, TypedDict


class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response.content}

#definindo o workflow
from langgraph.graph import START, StateGraph

graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

In [ ]:
result = graph.invoke({"question": "Como funcionam os carregadores goodwe?"})

print(f'Context: {result["context"]}\n\n')
print(f'Answer: {result["answer"]}')

In [ ]:
result